In [1]:
# Install library

In [2]:
%pip install sentence-transformers
%pip install -U datasets
%pip install -U "accelerate>=1.1.0"

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Prepare the data

In [4]:
"""
This examples show how to train a Cross-Encoder for the MS Marco dataset (https://github.com/microsoft/MSMARCO-Passage-Ranking).

In this example we use a knowledge distillation setup. Sebastian Hofstätter et al. trained in https://arxiv.org/abs/2010.02666
an ensemble of large Transformer models for the MS MARCO datasets and combines the scores from a BERT-base, BERT-large, and ALBERT-large model.

We use the logits scores from the ensemble to train a smaller model. We found that the MiniLM model gives the best performance while
offering the highest speed.

The resulting Cross-Encoder can then be used for passage re-ranking: You retrieve for example 100 passages
for a given query, for example with ElasticSearch, and pass the query+retrieved_passage to the CrossEncoder
for scoring. You sort the results then according to the output of the CrossEncoder.

This gives a significant boost compared to out-of-the-box ElasticSearch / BM25 ranking.

Running this script:
python train_cross-encoder-v2.py
"""
from torch.utils.data import DataLoader
from sentence_transformers import LoggingHandler, util
from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers.evaluation import SequentialEvaluator

from sentence_transformers import InputExample
import logging
from datetime import datetime
import gzip
import os
import tarfile
import tqdm
import torch
import json

# Added: limit how many docs are loaded into memory
MAX_DOCS = 1000000

util.http_get('https://github.com/arian-askari/ms-marco-MiniLM-L-12-v3/raw/main/train/CERerankingEvaluator_bm25cat.py', os.path.join('./', 'CERerankingEvaluator_bm25cat.py')) # download CERerankingEvaluator_bm25cat code which is a modification on CERerankingEvaluator.py class from SBERT
from CERerankingEvaluator_bm25cat import CERerankingEvaluator
#### Just some code to print debug information to stdout
logging.basicConfig(format='%(asctime)s - %(message)s',
                    datefmt='%Y-%m-%d %H:%M:%S',
                    level=logging.INFO,
                    handlers=[LoggingHandler()])
#### /print debug information to stdout

### We make a directory for storing the MS Marco dataset
data_folder = 'msmarco-data'
injection_folder = data_folder + "/injection_scores"
os.makedirs(data_folder, exist_ok=True)
os.makedirs(injection_folder, exist_ok=True)

# We download injection scores

train_scores_path = os.path.join(injection_folder, '1_bm25_scores_train_triples_small.json')
logging.info("Download "+os.path.basename(train_scores_path))
util.http_get('https://www.dropbox.com/scl/fi/ssgpoun44jtlrwy24wrad/1_bm25_scores_train_triples_small.json?rlkey=3og8ayxmyjxsei7okdumseaq7&raw=1', train_scores_path)

validation_scores_path = os.path.join(injection_folder, '5_bm25_scores_train-eval_triples.json')
logging.info("Download "+os.path.basename(validation_scores_path))
util.http_get('https://www.dropbox.com/scl/fi/q433llwfdk701x336ce3p/5_bm25_scores_train-eval_triples.json?rlkey=5782bylutyzmk10f1uax3iao5&raw=1', validation_scores_path)

# Loading injection scores and applying normalization (global min-max in the paper)
global_min_bm25 = 0
global_max_bm25 = 50
scores_path = "./msmarco-data/injection_scores/1_bm25_scores_train_triples_small.json"
scores = json.loads(open(scores_path, "r").read())
for qid in tqdm.tqdm(scores.keys(), desc = "reading scores...{}".format(scores_path)):
  for did, score in scores[qid].items():
    normalized_score = (score - global_min_bm25) / (global_max_bm25 - global_min_bm25)
    normalized_score = int(normalized_score * 100)
    scores[qid][did] = normalized_score

validation_scores_path = "./msmarco-data/injection_scores/5_bm25_scores_train-eval_triples.json"
scores_validation = json.loads(open(validation_scores_path, "r").read())
for qid in tqdm.tqdm(scores_validation.keys(), desc = "reading validation scores...{}".format(validation_scores_path)):
  if qid not in scores:
    scores[qid] = {}
  for did, score in scores_validation[qid].items():
    normalized_score = (score - global_min_bm25) / (global_max_bm25 - global_min_bm25)
    normalized_score = int(normalized_score * 100)
    scores[qid][did] = normalized_score

#First, we define the transformer model we want to fine-tune
model_name = 'microsoft/MiniLM-L12-H384-uncased'
train_batch_size = 8
num_epochs = 1
model_save_path = 'finetuned_CEs/train-cross-encoder-kd-bm25cat-'+model_name.replace("/", "-")+'-'+datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# Changed default_activation_function to activation_fn
model = CrossEncoder(model_name, num_labels=1, max_length=512, activation_fn=torch.nn.Identity())

#### Read the corpus files, that contain all the passages. Store them in the corpus dict
corpus = {}
collection_filepath = os.path.join(data_folder, 'collection.tsv')
if not os.path.exists(collection_filepath):
    tar_filepath = os.path.join(data_folder, 'collection.tar.gz')
    if not os.path.exists(tar_filepath):
        logging.info("Download collection.tar.gz")
        util.http_get('https://msmarco.blob.core.windows.net/msmarcoranking/collection.tar.gz', tar_filepath)

    with tarfile.open(tar_filepath, "r:gz") as tar:
        tar.extractall(path=data_folder)

with open(collection_filepath, 'r', encoding='utf8') as fIn:
    for i, line in enumerate(fIn):
        if i >= MAX_DOCS:
            break
        pid, passage = line.strip().split("\t")
        corpus[pid] = passage


### Read the train queries, store in queries dict
queries = {}
queries_filepath = os.path.join(data_folder, 'queries.train.tsv')
if not os.path.exists(queries_filepath):
    tar_filepath = os.path.join(data_folder, 'queries.tar.gz')
    if not os.path.exists(tar_filepath):
        logging.info("Download queries.tar.gz")
        util.http_get('https://msmarco.blob.core.windows.net/msmarcoranking/queries.tar.gz', tar_filepath)

    with tarfile.open(tar_filepath, "r:gz") as tar:
        tar.extractall(path=data_folder)


with open(queries_filepath, 'r', encoding='utf8') as fIn:
    for line in fIn:
        qid, query = line.strip().split("\t")
        queries[qid] = query


### Now we create our  dev data
train_samples = []
dev_samples = {}

# We use 200 random queries from the train set for evaluation during training
# Each query has at least one relevant and up to 200 irrelevant (negative) passages
num_dev_queries = 200
num_max_dev_negatives = 200

# msmarco-qidpidtriples.rnd-shuf.train-eval.tsv.gz and msmarco-qidpidtriples.rnd-shuf.train.tsv.gz is a randomly
# shuffled version of qidpidtriples.train.full.2.tsv.gz from the MS Marco website
# We extracted in the train-eval split 500 random queries that can be used for evaluation during training
train_eval_filepath = os.path.join(data_folder, 'msmarco-qidpidtriples.rnd-shuf.train-eval.tsv.gz')
if not os.path.exists(train_eval_filepath):
    logging.info("Download "+os.path.basename(train_eval_filepath))
    util.http_get('https://sbert.net/datasets/msmarco-qidpidtriples.rnd-shuf.train-eval.tsv.gz', train_eval_filepath)

with gzip.open(train_eval_filepath, 'rt') as fIn:
    for line in fIn:
        qid, pos_id, neg_id = line.strip().split()

        if pos_id not in corpus or neg_id not in corpus:
            continue
        if qid not in queries or qid not in scores:
            continue
        if pos_id not in scores[qid] or neg_id not in scores[qid]:
            continue

        if qid not in dev_samples and len(dev_samples) < num_dev_queries:
            dev_samples[qid] = {'query': list(), 'positive': list(), 'negative': list()}

        if qid in dev_samples:
            dev_samples[qid]['positive'].append(corpus[pos_id])
            dev_samples[qid]['query'].append("{} [SEP] {}".format(scores[qid][pos_id], queries[qid]))

            if len(dev_samples[qid]['negative']) < num_max_dev_negatives:
                dev_samples[qid]['negative'].append(corpus[neg_id])
                dev_samples[qid]['query'].append("{} [SEP] {}".format(scores[qid][neg_id], queries[qid]))


dev_qids = set(dev_samples.keys())

# Read our training file
# As input examples, we provide the (query, passage) pair together with the logits score from the teacher ensemble
teacher_logits_filepath = os.path.join(data_folder, 'bert_cat_ensemble_msmarcopassage_train_scores_ids.tsv')
train_samples = []
if not os.path.exists(teacher_logits_filepath):
    util.http_get('https://zenodo.org/record/4068216/files/bert_cat_ensemble_msmarcopassage_train_scores_ids.tsv?download=1', teacher_logits_filepath)

with open(teacher_logits_filepath) as fIn:
    for line in fIn:
        pos_score, neg_score, qid, pid1, pid2 = line.strip().split("\t")

        if qid in dev_qids:
            continue
        if pid1 not in corpus or pid2 not in corpus:
            continue
        if qid not in queries or qid not in scores:
            continue
        if pid1 not in scores[qid] or pid2 not in scores[qid]:
            continue

        train_samples.append(InputExample(texts=["{} [SEP] {}".format(scores[qid][pid1], queries[qid]), corpus[pid1]], label=float(pos_score)))
        train_samples.append(InputExample(texts=["{} [SEP] {}".format(scores[qid][pid2], queries[qid]), corpus[pid2]], label=float(neg_score)))

# We create a DataLoader to load our train samples
train_dataloader = DataLoader(train_samples, shuffle=True, batch_size=train_batch_size, drop_last=True)

# We add an evaluator, which evaluates the performance during training
# It performs a classification task and measures scores like F1 (finding relevant passages) and Average Precision
evaluator = SequentialEvaluator([
    CERerankingEvaluator(dev_samples, name='train-eval')
])

# Configure the training
warmup_steps = 5000
logging.info("Warmup-steps: {}".format(warmup_steps))

2026-04-01 21:15:33 - Download 1_bm25_scores_train_triples_small.json
2026-04-01 21:15:33 - HTTP Request: GET https://www.dropbox.com/scl/fi/ssgpoun44jtlrwy24wrad/1_bm25_scores_train_triples_small.json?rlkey=3og8ayxmyjxsei7okdumseaq7&raw=1 "HTTP/1.1 302 Found"
2026-04-01 21:15:34 - HTTP Request: GET https://uc6c511604ece901f1a41680f2ce.dl.dropboxusercontent.com/cd/0/inline/C9z50yakkjz0Y4RA-SjSPqmtkzV85Q9ep8Ne_nr8YOtu4SzUOrFAuMOQTsBVBVswSbDFKWjZcxTjjaKEZs26k8RXilKQNlvOvddhz9ENnqMLOn65ohZOyubwT3ui-QcuPCgd1dcYr1a83Gmuj76dL_6e/file# "HTTP/1.1 200 OK"


2026-04-01 21:17:26 - Download 5_bm25_scores_train-eval_triples.json
2026-04-01 21:17:26 - HTTP Request: GET https://www.dropbox.com/scl/fi/q433llwfdk701x336ce3p/5_bm25_scores_train-eval_triples.json?rlkey=5782bylutyzmk10f1uax3iao5&raw=1 "HTTP/1.1 302 Found"
2026-04-01 21:17:27 - HTTP Request: GET https://uc0b286f999a5ec12aad3d096e85.dl.dropboxusercontent.com/cd/0/inline/C9zbCTQ8HQdB57IVq7_W26ou82qmF0iiUFiqAtwyi0w_uz2R8ve5AJMLyuwoMZoX5bP1OZtGtTlTsvCcH193EXeAx4xZ467CK4LH02_If-xJJm8FqNN1ybaBbXB3Tj8VTR_84ko_0EqcI55NgQeZsibt/file# "HTTP/1.1 200 OK"


reading scores..../msmarco-data/injection_scores/1_bm25_scores_train_triples_small.json: 100%|████████████████████████████████████████████████████████████████████| 398791/398791 [00:26<00:00, 15283.67it/s]
reading validation scores..../msmarco-data/injection_scores/5_bm25_scores_train-eval_triples.json: 100%|█████████████████████████████████████████████████████████████████| 500/500 [00:00<00:00, 1889.84it/s]


2026-04-01 21:18:59 - HTTP Request: HEAD https://huggingface.co/microsoft/MiniLM-L12-H384-uncased/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-01 21:18:59 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/MiniLM-L12-H384-uncased/44acabbec0ef496f6dbc93adadea57f376b7c0ec/config.json "HTTP/1.1 200 OK"
2026-04-01 21:19:00 - HTTP Request: HEAD https://huggingface.co/microsoft/MiniLM-L12-H384-uncased/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-04-01 21:19:00 - HTTP Request: GET https://huggingface.co/api/models/microsoft/MiniLM-L12-H384-uncased "HTTP/1.1 200 OK"
2026-04-01 21:19:00 - HTTP Request: GET https://huggingface.co/api/models/microsoft/MiniLM-L12-H384-uncased/commits/main "HTTP/1.1 200 OK"
2026-04-01 21:19:00 - HTTP Request: GET https://huggingface.co/api/models/microsoft/MiniLM-L12-H384-uncased/discussions?p=0 "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: microsoft/MiniLM-L12-H384-uncased
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


2026-04-01 21:19:00 - HTTP Request: GET https://huggingface.co/api/models/microsoft/MiniLM-L12-H384-uncased/commits/refs%2Fpr%2F1 "HTTP/1.1 200 OK"
2026-04-01 21:19:01 - HTTP Request: HEAD https://huggingface.co/microsoft/MiniLM-L12-H384-uncased/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-01 21:19:01 - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-04-01 21:19:01 - HTTP Request: HEAD https://huggingface.co/microsoft/MiniLM-L12-H384-uncased/resolve/refs%2Fpr%2F1/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-04-01 21:19:01 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/MiniLM-L12-H384-uncased/44acabbec0ef496f6dbc93adadea57f376b7c0ec/config.json "HTTP/1.1 200 OK"
2026-04-01 21:19:01 - HTTP Request: HEAD https://huggingface.co/microsoft/MiniLM-L12-H384-uncased/resolve/refs%2Fpr%2F1/model.safetensors "HTTP/1.1 302 Found"
2026-04-

In [5]:
# Train

In [6]:
# Train the model
model.fit(train_dataloader=train_dataloader,
          loss_fct=torch.nn.MSELoss(),#
          evaluator=evaluator,
          epochs=num_epochs,
          evaluation_steps=5000,
          warmup_steps=warmup_steps,
          output_path=model_save_path,
          optimizer_params={'lr': 7e-6},
          use_amp=True)

#Save latest model
model.save(model_save_path+'-latest')

Step,Training Loss,Validation Loss,Evaluator 0,Sequential Score
5000,10.273886,No log,0.461497,0.461497
10000,4.541437,No log,0.521386,0.521386
15000,3.480463,No log,0.510587,0.510587
20000,3.179472,No log,0.550016,0.550016
25000,3.088283,No log,0.561000,0.561000
30000,2.824403,No log,0.560291,0.560291
35000,2.552297,No log,0.561497,0.561497
40000,2.305612,No log,0.552238,0.552238
45000,2.170031,No log,0.536312,0.536312
50000,2.156090,No log,0.528053,0.528053


2026-04-01 21:30:03 - CERerankingEvaluator: Evaluating the model on train-eval dataset in epoch 0.027790901258927827 after 5000 steps:
2026-04-01 21:30:24 - Queries: 75 	 Positives: Min 22.0, Mean 137.3, Max 441.0 	 Negatives: Min 22.0, Mean 128.8, Max 200.0
2026-04-01 21:30:24 - MRR@10: 46.15
2026-04-01 21:30:24 - Save model to finetuned_CEs/train-cross-encoder-kd-bm25cat-microsoft-MiniLM-L12-H384-uncased-2026-04-01_21-18-59


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-01 21:38:48 - CERerankingEvaluator: Evaluating the model on train-eval dataset in epoch 0.055581802517855654 after 10000 steps:
2026-04-01 21:39:10 - Queries: 75 	 Positives: Min 22.0, Mean 137.3, Max 441.0 	 Negatives: Min 22.0, Mean 128.8, Max 200.0
2026-04-01 21:39:10 - MRR@10: 52.14
2026-04-01 21:39:10 - Save model to finetuned_CEs/train-cross-encoder-kd-bm25cat-microsoft-MiniLM-L12-H384-uncased-2026-04-01_21-18-59


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-01 22:13:35 - CERerankingEvaluator: Evaluating the model on train-eval dataset in epoch 0.16674540755356695 after 30000 steps:
2026-04-01 22:13:56 - Queries: 75 	 Positives: Min 22.0, Mean 137.3, Max 441.0 	 Negatives: Min 22.0, Mean 128.8, Max 200.0
2026-04-01 22:13:56 - MRR@10: 56.03
2026-04-01 22:21:52 - CERerankingEvaluator: Evaluating the model on train-eval dataset in epoch 0.19453630881249478 after 35000 steps:
2026-04-01 22:22:12 - Queries: 75 	 Positives: Min 22.0, Mean 137.3, Max 441.0 	 Negatives: Min 22.0, Mean 128.8, Max 200.0
2026-04-01 22:22:12 - MRR@10: 56.15
2026-04-01 22:22:12 - Save model to finetuned_CEs/train-cross-encoder-kd-bm25cat-microsoft-MiniLM-L12-H384-uncased-2026-04-01_21-18-59


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-01 22:30:23 - CERerankingEvaluator: Evaluating the model on train-eval dataset in epoch 0.22232721007142262 after 40000 steps:
2026-04-01 22:30:43 - Queries: 75 	 Positives: Min 22.0, Mean 137.3, Max 441.0 	 Negatives: Min 22.0, Mean 128.8, Max 200.0
2026-04-01 22:30:43 - MRR@10: 55.22
2026-04-01 22:39:05 - CERerankingEvaluator: Evaluating the model on train-eval dataset in epoch 0.25011811133035045 after 45000 steps:
2026-04-01 22:39:26 - Queries: 75 	 Positives: Min 22.0, Mean 137.3, Max 441.0 	 Negatives: Min 22.0, Mean 128.8, Max 200.0
2026-04-01 22:39:26 - MRR@10: 53.63
2026-04-01 22:47:49 - CERerankingEvaluator: Evaluating the model on train-eval dataset in epoch 0.27790901258927825 after 50000 steps:
2026-04-01 22:48:11 - Queries: 75 	 Positives: Min 22.0, Mean 137.3, Max 441.0 	 Negatives: Min 22.0, Mean 128.8, Max 200.0
2026-04-01 22:48:11 - MRR@10: 52.81
2026-04-01 22:56:03 - CERerankingEvaluator: Evaluating the model on train-eval dataset in epoch 0.3056999138482061 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]